# Stage 8C — SurfaceField Energy-Scaled Optical Cockpit

**Engine:** Engine 1 (optical) → Engine 2 (exposure) bridge.  
**Model status:** `fluence_prediction` (optical fluence only).  
**Final export allowed:** `False`.

This notebook scales **real optical-field intensity arrays** to pulse fluence using the
Stage 8B energy ledger. It does **not** model absorption, nonlinear propagation, thermal
accumulation, material modification, ablation, voids, cracks, Δn, or waveguide formation.

It builds directly on Stage 8B (`01_optical_cockpit_energy_accounting.ipynb`):
Stage 8B gives `energy_at_sample_uJ`; Stage 8C uses the **actual simulated field**
(`bessel_twin_core` `SurfaceField` / propagation volume) instead of an assumed effective
beam area, and produces an energy-conserving transverse fluence map ("Mode B").

## 1. Controls and caveats

In [1]:
# --- Top-level controls (edit these) ---
planning_mode = True
save_outputs = False
figure_dpi = 180
show_caveats = True
show_diagnostic_panels = True

# Optical-field source selection
field_source_mode = "existing_repo_surfacefield"  # 'existing_repo_surfacefield' | 'existing_repo_volume'
field_source_path = None         # reserved: path to a saved field export (None = run engine live)
require_real_field = True        # if True, refuse to proceed without a real optical field
allow_synthetic_demo_field = False  # unit_test_or_demo_only; never a governed/saved output

# Engine case selection (used when running the repo engine live)
engine_preset = "fast"           # 'fast' | 'balanced' | 'publication'
engine_path = "ideal"           # 'ideal' | 'realistic'

# Laser / chain parameters (mirror Stage 8B)
wavelength_nm = 1030.0
pulse_duration_fs = 260.0
repetition_rate_Hz = 25_000.0
pulse_energy_before_optics_uJ = 200.0
average_power_limit_W = 10.0

slm_diffraction_efficiency = 0.75
selected_first_order_fraction = 0.73
relay_transmission = 0.90
objective_transmission = 0.85
sample_interface_transmission = 0.95

In [2]:
# --- Save guard: diagnostic outputs must carry their caveats ---
if save_outputs and not show_caveats:
    raise ValueError(
        "Cannot save Stage 8C outputs with show_caveats=False. "
        "Diagnostic fluence outputs must carry their caveats."
    )

CAVEATS = (
    "This notebook scales real optical-field intensity arrays to pulse fluence using the "
    "Stage 8B energy ledger.\n"
    "It does not model absorption, nonlinear propagation, thermal accumulation, material "
    "modification, ablation, voids, cracks, \u0394n, or waveguide formation.\n\n"
    "Energy-scaled fluence maps are OPTICAL fluence predictions only: not absorbed-energy "
    "maps, not dose maps, not material-modification maps, and not damage predictions."
)
if show_caveats:
    print(CAVEATS)

This notebook scales real optical-field intensity arrays to pulse fluence using the Stage 8B energy ledger.
It does not model absorption, nonlinear propagation, thermal accumulation, material modification, ablation, voids, cracks, Δn, or waveguide formation.

Energy-scaled fluence maps are OPTICAL fluence predictions only: not absorbed-energy maps, not dose maps, not material-modification maps, and not damage predictions.


## 2. Energy ledger from Stage 8B

In [3]:
import sys, os
import numpy as np

# Ensure Publication_Study root is importable (bessel_twin_core, vbb_study)
_here = os.getcwd()
for _up in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
    if os.path.isfile(os.path.join(_up, "bessel_twin_core.py")) and _up not in sys.path:
        sys.path.insert(0, _up)

from vbb_study.digital_twin.energy_accounting import (
    compute_energy_ledger, default_holographic_chain,
)
from vbb_study.digital_twin.field_coupling import (
    MissingOpticalFieldError,
    extract_plane_from_surfacefield, extract_stack_from_surfacefield,
    plane_from_arrays,
)
from vbb_study.digital_twin.field_fluence import (
    scale_plane_to_fluence, scale_stack_to_fluence,
    peak_intensity_from_fluence_result, field_fluence_summary,
)
from vbb_study.digital_twin.field_figures import plot_stage8c_field_fluence_preview

chain = default_holographic_chain(
    slm_diffraction_efficiency=slm_diffraction_efficiency,
    selected_first_order_fraction=selected_first_order_fraction,
    relay_transmission=relay_transmission,
    objective_transmission=objective_transmission,
    sample_interface_transmission=sample_interface_transmission,
)
ledger = compute_energy_ledger(
    pulse_energy_before_optics_uJ, repetition_rate_Hz, chain,
    average_power_limit_W=average_power_limit_W,
)
print(f"Energy at sample: {ledger.energy_at_sample_uJ:.4g} \u00b5J")
print(f"Average power at sample: {ledger.average_power_at_sample_W*1e3:.4g} mW")
for w in ledger.ledger_warnings:
    print("  warning:", w)

Energy at sample: 79.58 µJ
Average power at sample: 1989 mW


## 3. Load the real existing-repo optical field

We run the repository engine (`bessel_twin_core.run_case`) live to obtain a **real**
`SurfaceField` and propagation volume. No analytic placeholder beam is fabricated in the
production path. If no real field is available and the synthetic demo is disabled, we fail
loudly.

In [4]:
surface_field = None
volume = None
field_origin = "none"

try:
    import bessel_twin_core as bt
    _res = bt.run_case(preset=engine_preset, path=engine_path, case_id="stage8c_cockpit")
    surface_field = _res.get("surface_field")
    volume = _res.get("volume")
    field_origin = f"bessel_twin_core.run_case(preset={engine_preset!r}, path={engine_path!r})"
except Exception as exc:  # engine unavailable in this environment
    print("Could not run engine live:", exc)

if surface_field is None and volume is None:
    if require_real_field and not allow_synthetic_demo_field:
        raise MissingOpticalFieldError(
            "No real optical field available and synthetic demo is disabled "
            "(allow_synthetic_demo_field=False). Refusing to fabricate a placeholder beam."
        )
    elif allow_synthetic_demo_field:
        # unit_test_or_demo_only — NEVER a governed/saved output.
        print("[unit_test_or_demo_only] building a labelled demo field; will not be saved as governed output.")
        _n = 128
        _x = (np.arange(_n) - _n / 2) * 0.25
        _X, _Y = np.meshgrid(_x, _x, indexing="xy")
        _R = np.hypot(_X, _Y)
        _demo = np.exp(-(_R ** 2) / (2 * 2.0 ** 2))
        demo_plane = plane_from_arrays(
            _demo, dx_um=0.25, dy_um=0.25, z_um=0.0,
            field_label="demo_gaussian", source_status="unit_test_or_demo_only",
        )
        save_outputs = False  # force: demo never saved

else:
    print("Loaded real field from:", field_origin)

Loaded real field from: bessel_twin_core.run_case(preset='fast', path='ideal')


## 4. Convert to a canonical plane / stack

In [5]:
plane = None
stack = None

if surface_field is not None:
    plane = extract_plane_from_surfacefield(surface_field)
    print("plane:", plane.intensity.shape, f"dx={plane.dx_um:.4g} \u00b5m",
          f"z={plane.z_um} \u00b5m", plane.source_status)
elif allow_synthetic_demo_field and 'demo_plane' in dir():
    plane = demo_plane
    print("plane (demo):", plane.intensity.shape, plane.source_status)

if field_source_mode == "existing_repo_volume" and volume is not None:
    stack = extract_stack_from_surfacefield(volume)
    print("stack:", stack.intensity_zyx.shape, f"dx={stack.dx_um:.4g} \u00b5m",
          f"nz={stack.z_um.size}", stack.source_status)
elif volume is not None:
    # Always make the stack available for diagnostics when we have a real volume.
    stack = extract_stack_from_surfacefield(volume)
    print("stack (diagnostic):", stack.intensity_zyx.shape, f"nz={stack.z_um.size}")

plane: (512, 512) dx=0.25 µm z=84.5001212544426 µm real_optical_field
stack (diagnostic): (41, 192, 192) nz=41


## 5. Scale field to fluence (energy-conserving, Mode B)

In [6]:
plane_result = None
stack_result = None

if plane is not None:
    plane_result = scale_plane_to_fluence(plane, ledger.energy_at_sample_uJ)
if stack is not None:
    stack_result = scale_stack_to_fluence(stack, ledger.energy_at_sample_uJ)

## 6. Report

In [7]:
if plane is not None:
    s = field_fluence_summary(plane, ledger.energy_at_sample_uJ, pulse_duration_fs)
    print("--- Plane fluence summary ---")
    print(f"  energy at sample      : {ledger.energy_at_sample_uJ:.4g} \u00b5J")
    print(f"  integrated energy     : {s['integrated_energy_uJ']:.6g} \u00b5J")
    print(f"  energy residual       : {s['energy_conservation_residual_uJ']:.3e} \u00b5J")
    print(f"  peak fluence          : {s['peak_fluence_j_cm2']:.4g} J/cm\u00b2")
    print(f"  peak intensity (approx): {s['peak_intensity_w_cm2']:.3e} W/cm\u00b2")
    print(f"  model_status          : {s['model_status']}  (final_export_allowed={s['final_export_allowed']})")

if stack_result is not None:
    ss = field_fluence_summary(stack, ledger.energy_at_sample_uJ, pulse_duration_fs)
    print("\n--- Stack fluence summary ---")
    print(f"  planes                : {ss['n_planes']}")
    print(f"  peak fluence          : {ss['peak_fluence_j_cm2']:.4g} J/cm\u00b2 @ z={ss['peak_z_um']:.4g} \u00b5m")
    print(f"  propagation drift     : {ss['propagation_energy_drift_fraction']:.3%}")
    print(f"  max transverse E resid: {ss['max_transverse_energy_residual_uJ']:.3e} \u00b5J")
    if ss['propagation_energy_drift_fraction'] > 0.2:
        print("  warning: large transverse-power drift across z (crop window captures a "
              "shrinking fraction of the field away from the Bessel zone).")

if show_caveats:
    print("\n" + CAVEATS)

--- Plane fluence summary ---
  energy at sample      : 79.58 µJ
  integrated energy     : 79.5791 µJ
  energy residual       : 0.000e+00 µJ
  peak fluence          : 71.16 J/cm²
  peak intensity (approx): 2.737e+14 W/cm²
  model_status          : fluence_prediction  (final_export_allowed=False)

--- Stack fluence summary ---
  planes                : 41
  peak fluence          : 133 J/cm² @ z=-103 µm
  propagation drift     : 99.800%
  max transverse E resid: 2.842e-14 µJ

This notebook scales real optical-field intensity arrays to pulse fluence using the Stage 8B energy ledger.
It does not model absorption, nonlinear propagation, thermal accumulation, material modification, ablation, voids, cracks, Δn, or waveguide formation.

Energy-scaled fluence maps are OPTICAL fluence predictions only: not absorbed-energy maps, not dose maps, not material-modification maps, and not damage predictions.


## 7. Diagnostic figure

In [8]:
fig = None
if show_diagnostic_panels:
    target = stack if stack is not None else plane
    target_result = stack_result if stack_result is not None else plane_result
    if target is not None and target_result is not None:
        fig = plot_stage8c_field_fluence_preview(
            target, target_result,
            energy_ledger=ledger, pulse_duration_fs=pulse_duration_fs,
            show_caveats=show_caveats, dpi=figure_dpi,
        )
        print("figure metadata:", fig.stage8c_metadata)

figure metadata: {'stage': 'stage8c_surfacefield_energy_scaled_cockpit', 'figure_status': 'diagnostic_allowed', 'model_status': 'fluence_prediction', 'final_export_allowed': 'False', 'source_status': 'real_optical_field'}


## 8. Save (only with caveats enabled)

In [ ]:
import csv
from pathlib import Path

if save_outputs:
    if not show_caveats:
        raise ValueError("Refusing to save with show_caveats=False.")
    # Governance: never save a non-real (synthetic/demo) field as a governed output.
    governed = (plane is None or plane.is_governed_source) and (stack is None or stack.is_governed_source)
    if not governed:
        raise ValueError("Refusing to save: a non-governed (synthetic/demo) field is in use.")

    csv_dir = Path("outputs/csv/digital_twin")
    csv_dir.mkdir(parents=True, exist_ok=True)
    fig_dir = Path("outputs/figures/digital_twin")
    fig_dir.mkdir(parents=True, exist_ok=True)

    summary = field_fluence_summary(
        stack if stack is not None else plane, ledger.energy_at_sample_uJ, pulse_duration_fs,
    )
    csv_path = csv_dir / "stage8c_field_fluence_summary_example.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["key", "value"])
        for k, v in summary.items():
            w.writerow([k, v])
    print("wrote", csv_path)

    if fig is not None:
        fig_path = fig_dir / "stage8c_surfacefield_energy_scaled_preview.png"
        target = stack if stack is not None else plane
        target_result = stack_result if stack_result is not None else plane_result
        plot_stage8c_field_fluence_preview(
            target, target_result, energy_ledger=ledger,
            pulse_duration_fs=pulse_duration_fs, output_path=fig_path,
            show_caveats=show_caveats, dpi=figure_dpi,
        )
        print("wrote", fig_path)
else:
    print("save_outputs=False — nothing written.")

save_outputs=False — nothing written.


: 